In [ ]:
import matplotlib.pyplot as plt
import rasterio
import numpy as np
import os
import json
from pyproj import Transformer
import rpcm
from scipy import interpolate
import rasterio.transform
import torch
from torch_scatter import scatter_max

In [ ]:
# Function to save geo tif
def save_geotiff(data, transform, crs, filename):
    # Create the file
    with rasterio.open(filename, 'w', driver='GTiff', height=data.shape[0], width=data.shape[1], count=1, dtype=data.dtype, crs=crs, transform=transform) as dst:
        dst.write(data, 1)

In [ ]:
city = "JAX"
tile = "370"
crop = "5"
dataset_dir = "/mnt/adisk/datasets/shadow_dataset"
tmp_dir = "tmp"

os.makedirs(tmp_dir, exist_ok=True)

# Load JSON metadata
json_path = os.path.join(dataset_dir, "root_dir", f"{city}_{tile}", f"{city}_{tile}_{crop}.json")
with open(json_path) as f:
    json_meta = json.load(f)

# Load image
img_path = os.path.join(dataset_dir, "crops", f"{city}_{tile}", f"{city}_{tile}_{crop}.tif")
with rasterio.open(img_path) as src:
    img = src.read().squeeze()
    img_meta = src.meta

# Load DSM
# if not using res 1.0,  we need to scale the dsm so the 2d grid is in the same scale as the dsm height
# res = "10"
res = "05"
dsm_path = os.path.join(dataset_dir, "dsm", f"{city}_{tile}", f"{city}_{tile}_resolution_{res}_2017.tif")
with rasterio.open(dsm_path) as src:
    dsm = src.read(masked=True)
    mask = dsm.mask
    dsm_data = dsm.data
    dsm_data[mask] = np.nan
    dsm_data[mask] = np.nanmedian(dsm_data)
    dsm = dsm_data.squeeze()
    # Apply a nan median filter to the dsm
    import scipy.ndimage
    dsm = scipy.ndimage.median_filter(dsm, size=3)
    meta = src.meta
    dsm /= meta["transform"][0]

In [ ]:
# Plot dsm and image
fig, ax = plt.subplots(1, 2, figsize=(20, 10))
ax[0].imshow(dsm, cmap="gist_earth")
plt.colorbar(ax[0].imshow(dsm, cmap="gist_earth"), ax=ax[0])
ax[1].imshow(img, cmap="gray")
plt.show()

In [ ]:
### RASTER IO RESAMPLE

from rasterio.enums import Resampling
import scipy.ndimage

def pixel_to_geo(coords, dsm_transform):
    """Convert pixel coordinates to geographic coordinates using DSM metadata."""
    return np.array(rasterio.transform.xy(dsm_transform, coords[:, 0], coords[:, 1]))

def project_shadows(img_path, lon, lat, z):
    """Project shadows using RPC model."""
    return rpcm.projection(img_path, lon, lat, z)

def upsample_dsm(dsm_path, upscale_factor=4):
    """Upsample DSM using pixel-as-area approach."""
    src = rasterio.open(dsm_path)
    meta = src.meta.copy()

    # Calculate new dimensions
    original_height, original_width = src.shape
    new_height = int(original_height * upscale_factor)
    new_width = int(original_width * upscale_factor)

    # Calculate the pixel size in the new resolution
    pixel_size_x = src.transform[0] / upscale_factor
    pixel_size_y = abs(src.transform[4]) / upscale_factor

    # Explicitly recalculate the transform to ensure alignment with the original grid
    transform_upscaled = rasterio.transform.from_origin(
        src.transform.c,  # x coordinate of the upper-left corner
        src.transform.f,  # y coordinate of the upper-left corner
        pixel_size_x, pixel_size_y  # new pixel size
    )

    # Update metadata
    meta.update({
        'height': new_height,
        'width': new_width,
        'transform': transform_upscaled
    })

    # Resample DSM using 'average' method for smoother results
    dsm_upscaled = src.read(
        out_shape=(1, new_height, new_width),
        resampling=Resampling.bilinear, masked=True
    )

    # Handle NaN values by filling with the median of the surrounding valid data
    dsm_upscaled_data = dsm_upscaled.data.squeeze()
    dsm_upscaled_mask = dsm_upscaled.mask.squeeze()
    dsm_upscaled_data[dsm_upscaled_mask] = np.nanmedian(dsm_upscaled_data)
    dsm_upscaled_data = scipy.ndimage.median_filter(dsm_upscaled_data, size=3)

    # Create UTM coordinates for the upscaled grid, adjusted for pixel centers
    x = np.linspace(transform_upscaled[2] + pixel_size_x / 2, 
                    transform_upscaled[2] + pixel_size_x * new_width - pixel_size_x / 2, 
                    new_width)
    y = np.linspace(transform_upscaled[5] - pixel_size_y / 2, 
                    transform_upscaled[5] - pixel_size_y * new_height + pixel_size_y / 2, 
                    new_height)

    xy_utm_upscaled = np.array(np.meshgrid(x, y))

    src.close()

    return dsm_upscaled_data, xy_utm_upscaled, meta, transform_upscaled

# Upscale DSM and get UTM coordinates
upscale_factor = 4
dsm_upscaled, xy_utm_upscaled, meta_upscaled, transform_upscaled = upsample_dsm(dsm_path, upscale_factor)

# Scale the height values to the same scale as the DSM coordinates
dsm_upscaled = dsm_upscaled / meta_upscaled["transform"][0]

# Save upscaled DSM
upscaled_dsm_path = os.path.join(tmp_dir, f"{city}_{tile}_{crop}_upscaled_dsm.tif")
with rasterio.open(upscaled_dsm_path, 'w', **meta_upscaled) as dst:
    dst.write(dsm_upscaled, 1)

In [ ]:
### NUMPY RESAMPLE


# def pixel_to_geo(coords, dsm_transform):
#     """Convert pixel coordinates to geographic coordinates using DSM metadata."""
#     return np.array(rasterio.transform.xy(dsm_transform, coords[:, 0], coords[:, 1]))

# def project_shadows(img_path, lon, lat, z):
#     """Project shadows using RPC model."""
#     return rpcm.projection(img_path, lon, lat, z)

# def upsample_array(arr, factor=4):
#     if arr.ndim == 2:
#         y, x = arr.shape
#         z = 1
#     elif arr.ndim == 3:
#         z, y, x = arr.shape

#     # Original grid
#     x_orig = np.arange(x)
#     y_orig = np.arange(y)
    
#     # New grid (4 times denser)
#     x_new = np.linspace(0, x - 1, x * factor)
#     y_new = np.linspace(0, y - 1, y * factor)
    
#     # Create the new coordinate matrices
#     xx_new, yy_new = np.meshgrid(x_new, y_new)
    
#     # Perform interpolation
#     if z == 1:
#         # For 2D array (e.g., DSM)
#         interp_func = interpolate.RegularGridInterpolator((y_orig, x_orig), arr, method='linear', bounds_error=False, fill_value=None)
#         result = interp_func((yy_new, xx_new))
#     else:
#         # For 3D array (e.g., easting and northing)
#         result = np.zeros((z, y * factor, x * factor))
#         for i in range(z):
#             interp_func = interpolate.RegularGridInterpolator((y_orig, x_orig), arr[i], method='linear', bounds_error=False, fill_value=None)
#             result[i] = interp_func((yy_new, xx_new))
    
#     return result


# # Upscale both dsm and coords to oversample with more detail
# upscale_factor = 4
# # Get all utm coordinates for the dsm
# x = np.array(range(dsm.shape[1]))
# y = np.array(range(dsm.shape[0]))
# xy = np.array(np.meshgrid(x, y))
# xy_utm = np.array(rasterio.transform.xy(meta["transform"], xy[1], xy[0]))
# # Upsample the eastings and northings by linear interpolation
# xy_utm_upscaled = upsample_array(xy_utm, upscale_factor)
# # Upsample the DSM by linear interpolation
# dsm_upscaled = upsample_array(dsm, upscale_factor)
# # Scale the height so the 2D grid is in the same scale as the DSM height
# dsm_upscaled *= upscale_factor

# # Save upscaled DSM
# upscaled_dsm_path = os.path.join(tmp_dir, f"{city}_{tile}_{crop}_upscaled_dsm.tif")
# save_geotiff(dsm_upscaled, rasterio.transform.from_origin(xy_utm_upscaled[0, 0, 0], xy_utm_upscaled[1, 0, 0], meta["transform"][0], meta["transform"][4]), meta["crs"], upscaled_dsm_path)

In [ ]:
# Plot both dsms
fig, ax = plt.subplots(1, 2, figsize=(20, 10))
ax[0].imshow(dsm, cmap="gist_earth")
ax[1].imshow(dsm_upscaled, cmap="gist_earth")
plt.show()

In [ ]:
# Shadow projection on the DSM

#### p - - >
#### q
#### |
#### |
#### v

# a slope

import os 
import math


def calculate_shadow_params(elevation, azimuth):
    elevation_rad = math.radians(float(elevation))
    azimuth_rad = math.radians(float(azimuth))

    p = -math.sin(azimuth_rad) * math.cos(elevation_rad)
    q = math.cos(azimuth_rad) * math.cos(elevation_rad)

    a = math.tan(elevation_rad)

    return p, q, a

p, q, a = calculate_shadow_params(json_meta["sun_elevation"], json_meta["sun_azimuth"])

shadow_dsm_path = os.path.join(tmp_dir, f"{city}_{tile}_{crop}_shadow.tif")

os.system(
    f"shadowcast {p} {q} {a} {upscaled_dsm_path} {shadow_dsm_path}"
)

In [ ]:
# Open shadow .tif
with rasterio.open(shadow_dsm_path) as src:
    shadow = src.read()
    shadow = shadow.squeeze()

missing_dsm_values = np.isnan(dsm_upscaled)
shadow[missing_dsm_values] = 1
shadow = np.nan_to_num(shadow)
shadow[shadow != 0] = 1

# Convert back the upscaled dsm to the original scale
dsm_upscaled *= meta_upscaled["transform"][0]

# Plot shadow along with dsm and image
fig, ax = plt.subplots(1, 3, figsize=(20, 10))
cax = ax[0].imshow(dsm_upscaled, cmap="gist_earth")
ax[0].set_title("DSM")
fig.colorbar(cax, ax=ax[0], orientation='vertical', fraction=0.046, pad=0.04)
ax[1].imshow(img, cmap="gray")
ax[1].set_title("Image")
ax[2].imshow(shadow.squeeze(), cmap="gray")
ax[2].set_title("Shadow on DSM")
plt.show()

In [ ]:
# Transform DSM UTM coordinates to lat/lon
transformer = Transformer.from_crs("epsg:32617", "epsg:4326", always_xy=True)
lon, lat = transformer.transform(xy_utm_upscaled[0], xy_utm_upscaled[1])

# Compute the x,y coordinates in the image
x_img, y_img = rpcm.projection(img_path, lon.flatten(), lat.flatten(), dsm_upscaled.flatten()) # x is cols, y is rows

# Flatten arrays
x_img = np.round(x_img).flatten().astype(int) # col coordinate of all the pixels
y_img = np.round(y_img).flatten().astype(int) # row coordinate of all the pixels (some of them are outside the image)
dsm_flat = dsm_upscaled.flatten()
shadow_flat = shadow.flatten()

# Filter x_img and y_img that have valid coordinates in the image
valid = (x_img >= 0) & (x_img < img.shape[1]) & (y_img >= 0) & (y_img < img.shape[0])
x_img = x_img[valid]
y_img = y_img[valid]
dsm_flat = dsm_flat[valid]
shadow_flat = shadow_flat[valid]

# Now we will project shadows from the DSM to the image
# Note that there might be multiple DSM values projecting to the same pixel in the image for some pixels
# For each pixel in the image, we want to find the max DSM value that projects to that pixel and use its shadow value to fill the shadow image
index = y_img * img.shape[1] + x_img # unique index for each pixel coordinate in the image
tensor_index = torch.tensor(index)
tensor_dsm = torch.tensor(dsm_flat)

# Filter out unambiguous pixels, i.e. pixels that only have one DSM value projecting to them
unique_index, unique_counts = np.unique(index, return_counts=True)
unique_index = unique_index[unique_counts == 1]
unique_mask = np.isin(index, unique_index)

# Shadow projection from the DSM to the image
shadow_image = np.full_like(img, np.nan)

# For unambiguous pixels, we can directly assign the shadow value
unambiguous_x = x_img[unique_mask]
unambiguous_y = y_img[unique_mask]
unambiguous_shadow = shadow_flat[unique_mask]
shadow_image[unambiguous_y, unambiguous_x] = unambiguous_shadow

# For ambiguous pixels we will create a z buffer using scatter max, for each ambiguous pixel we project the shadow value of the max DSM value
# that projects to that pixel
ambiguous_index = tensor_index[~unique_mask]
ambiguous_dsm = tensor_dsm[~unique_mask]
ambiguous_shadow = shadow_flat[~unique_mask]
# scatter_max creates an output tensor (z_buffer) with a size based on the maximum index value in ambiguous_index.
# The size of z_buffer will be the maximum index value in ambiguous_index + 1, because scatter_max allocates space for all possible indices up to that maximum index.
z_buffer, z_buffer_index = scatter_max(ambiguous_dsm, ambiguous_index)
# Extract the unique indices of ambiguous pixels (they represent the position in the shadow_image)
ambiguous_x = x_img[~unique_mask]
ambiguous_y = y_img[~unique_mask]

# Calculate the indices of the ambiguous pixels in the image to correctly update the shadow_image
ambiguous_pixel_indices = ambiguous_y * img.shape[1] + ambiguous_x

# Now filter the z_buffer and z_buffer_index to only keep values for the ambiguous pixels
# Only consider pixels that fall within the bounds of the image
z_buffer_filtered = z_buffer[ambiguous_pixel_indices]
z_buffer_index_filtered = z_buffer_index[ambiguous_pixel_indices]

# Now we need to retrieve the corresponding shadow values for the max DSM values
max_shadow = ambiguous_shadow[z_buffer_index_filtered]

# Fill the shadow image for the ambiguous pixels with the max shadow value
shadow_image[ambiguous_y, ambiguous_x] = max_shadow

print(f"Number of nan values in shadow image: {np.isnan(shadow_image).sum()}")

In [ ]:
# Plot shadow image and image
fig, ax = plt.subplots(1, 2, figsize=(20, 10))
ax[0].imshow(img, cmap="gray")
ax[1].imshow(shadow_image, cmap="gray")

In [ ]:
output_dir = os.path.join(dataset_dir, "shadows", f"{city}_{tile}")
os.makedirs(output_dir, exist_ok=True)

shadow_path = os.path.join(output_dir, f"{city}_{tile}_{crop}.tif")

# Save shadow image
save_geotiff(shadow_image, img_meta["transform"], img_meta["crs"], shadow_path)

## old code below

In [ ]:
def get_shadow_coords_dsm(shadow_mask):
    """Extract pixels that should be painted as shadow."""
    shadow_mask_bool = shadow_mask == 0
    coords = np.argwhere(shadow_mask_bool)
    return coords,

def pixel_to_geo(coords, dsm_transform):
    """Convert pixel coordinates to geographic coordinates using DSM metadata."""
    return np.array(rasterio.transform.xy(dsm_transform, coords[:, 0], coords[:, 1]))

def project_shadows(img_path, lon, lat, z):
    """Project shadows using RPC model."""
    return rpcm.projection(img_path, lon, lat, z)

def paint_shadows(img, shadow_coords):
    """Paint shadowed areas in red with improved visibility."""
    # Ensure img is 2D (single channel)
    if img.ndim > 2:
        img = img.squeeze()
    
    # Normalize the image to 0-1 range
    img_normalized = img.astype(float) / np.max(img)
    
    # Create RGB image
    img_rgb = np.stack((img_normalized,) * 3, axis=-1)
    
    # Unpack x and y coordinates
    x, y = shadow_coords
    
    # Round coordinates and convert to integers
    x_rounded = np.round(x).astype(int)
    y_rounded = np.round(y).astype(int)
    
    # Clip coordinates to image boundaries
    x_clipped = np.clip(x_rounded, 0, img.shape[1] - 1)
    y_clipped = np.clip(y_rounded, 0, img.shape[0] - 1)
    
    # Paint shadows (red with increased visibility)
    img_rgb[y_clipped, x_clipped, 0] = 1.0  # Red channel
    img_rgb[y_clipped, x_clipped, 1] *= 0.5  # Reduce green
    img_rgb[y_clipped, x_clipped, 2] *= 0.5  # Reduce blue
    
    # Adjust overall brightness
    img_rgb = np.power(img_rgb, 0.8)  # Gamma correction to increase brightness
    
    return (img_rgb * 255).astype(np.uint8)

def plot_results(dsm, img_with_shadows, shadow):
    """
    Plot DSM, image with shadows, original shadow mask, and new shadow mask.
    The new shadow mask is white everywhere except for the projected shadow areas, which are black.
    """
    fig, axes = plt.subplots(1, 3, figsize=(20, 10))
    
    # Plot DSM
    axes[0].imshow(dsm.squeeze(), cmap="gist_earth")
    axes[0].set_title("DSM")
    
    # Plot image with shadows
    axes[1].imshow(img_with_shadows)
    axes[1].set_title("Image with Projected Shadows")
    
    # Plot original shadow mask
    axes[2].imshow(shadow.squeeze(), cmap="gray")
    axes[2].set_title("Shadow Mask from the DSM")
    
    plt.tight_layout()
    plt.show()


# Pixel coordinates of the shadows in the DSM
coords = get_shadow_coords_dsm(shadow)[0]

# Convert pixel coordinates to geographic coordinates
utm_coords = pixel_to_geo(coords, meta["transform"])

# Convert UTM coordinates to lat/lon
transformer = Transformer.from_crs("epsg:32617", "epsg:4326", always_xy=True)
lon, lat = transformer.transform(utm_coords[0], utm_coords[1])

# Project the shadows using the RPC model on the image
# TODO: at this momment we still haven't done bundle adjustment, so the shadows are not accurate
# TODO: double check altitude reference system
x = coords[:, 0]
y = coords[:, 1]
z = dsm[y, x].squeeze()
shadow_coords = project_shadows(img_path, lon, lat, z)

img_with_shadows = paint_shadows(img, shadow_coords)

plot_results(dsm, img_with_shadows, shadow)

In [ ]:
def create_binary_shadow_mask(img_shape, shadow_coords):
    """
    Create a binary mask from shadow coordinates.
    
    Parameters:
    img_shape (tuple): The shape of the image (height, width).
    shadow_coords (tuple): The coordinates of shadow points (x, y).
    
    Returns:
    np.ndarray: Binary mask of the same shape as the image.
    """
    # Initialize an empty binary mask with zeros
    binary_mask = np.zeros(img_shape, dtype=np.uint8)
    
    # Unpack shadow coordinates (x, y)
    x, y = shadow_coords
    
    # Round coordinates and convert to integers
    x_rounded = np.round(x).astype(int)
    y_rounded = np.round(y).astype(int)
    
    # Clip coordinates to image boundaries
    x_clipped = np.clip(x_rounded, 0, img_shape[1] - 1)  # Width (x)
    y_clipped = np.clip(y_rounded, 0, img_shape[0] - 1)  # Height (y)
    
    # Set the shadowed pixels to 1 (shadows)
    binary_mask[y_clipped, x_clipped] = 1
    
    return binary_mask

# Use the function to create the binary shadow mask
binary_shadow_mask = create_binary_shadow_mask(img.squeeze().shape, shadow_coords)

# Visualize the binary mask
plt.imshow(binary_shadow_mask, cmap="gray")
plt.title("Binary Shadow Mask")
plt.show()


In [ ]:
# save shadow mask as tif
shadow_mask_path = "shadow_binary.tif"
with rasterio.open(
    shadow_mask_path,
    "w",
    driver="GTiff",
    height=binary_shadow_mask.shape[0],
    width=binary_shadow_mask.shape[1],
    count=1,
    dtype=binary_shadow_mask.dtype,
    crs=src.crs,
    transform=src.transform,
) as dst:
    dst.write(binary_shadow_mask, 1)

In [ ]:
from scipy.ndimage import binary_dilation

# Dilate the binary mask to highlight shadowed areas more
dilated_mask = binary_dilation(binary_shadow_mask, iterations=3)

# Plot the dilated binary mask
plt.imshow(dilated_mask, cmap="gray", vmin=0, vmax=1)
plt.title("Dilated Binary Shadow Mask")
plt.show()
